## Import Library

In [ ]:
"""Per-class F1 threshold optimization for the trained DenseNet-121.
 
Loads multilabel_model.pt, runs inference on val set, finds the threshold
per class that maximizes F1, then evaluates on test set.
 
Output: /kaggle/working/multilabel_thresholds.json
 
Reference: Strick et al. (2025) — per-class threshold tuning approach.
"""
 
import json
import numpy as np
from pathlib import Path
 
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score
import timm
 
from dataset_utils import (
    ALL_CONDITIONS,
    build_image_index,
    load_multilabel_csv,
    patient_level_split,
    ChestXrayDataset,
    get_eval_transform,
)

## Config

In [ ]:
# Config 
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE     = 32
NUM_WORKERS    = 2
MODEL_PATH     = Path("/kaggle/working/multilabel_model.pt")
THRESHOLD_PATH = Path("/kaggle/working/multilabel_thresholds.json")

## Helper Function

In [ ]:
def load_model() -> torch.nn.Module:
    """Load trained DenseNet-121 weights into eval mode."""
    model = timm.create_model(
        "densenet121", pretrained=False, num_classes=14, drop_rate=0.2
    )
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model
 
 
def collect_probs(model, loader) -> tuple[np.ndarray, np.ndarray]:
    """Run inference and return (all_probs, all_labels) as pre-allocated arrays."""
    n = len(loader.dataset)
    all_probs = np.empty((n, len(ALL_CONDITIONS)), dtype=np.float32)
    all_labels = np.empty((n, len(ALL_CONDITIONS)), dtype=np.float32)
    idx = 0
 
    with torch.no_grad():
        for images, labels in loader:
            bsz = images.size(0)
            probs = torch.sigmoid(model(images.to(DEVICE))).float().cpu().numpy()
            all_probs[idx:idx+bsz]  = probs
            all_labels[idx:idx+bsz] = labels.numpy()
            idx += bsz
 
    return all_probs, all_labels
 
 
def find_best_thresholds(all_probs: np.ndarray, all_labels: np.ndarray) -> list[float]:
    """Find per-class F1-optimal threshold by grid search over [0.10, 0.85]."""
    thresholds = []
    for i in range(len(ALL_CONDITIONS)):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.10, 0.90, 0.05):
            score = f1_score(
                all_labels[:, i],
                (all_probs[:, i] >= t).astype(int),
                zero_division=0,
            )
            if score > best_f1:
                best_f1, best_t = score, float(t)
        thresholds.append(round(best_t, 2))
    return thresholds
 
 
def evaluate_test(
    all_probs: np.ndarray,
    all_labels: np.ndarray,
    thresholds: list[float],
) -> None:
    """Print per-class and mean AUC/F1 on test set."""
    aucs, f1s = [], []
    for i, cond in enumerate(ALL_CONDITIONS):
        if all_labels[:, i].sum() == 0:
            continue
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        f1 = f1_score(
            all_labels[:, i],
            (all_probs[:, i] >= thresholds[i]).astype(int),
            zero_division=0,
        )
        aucs.append(auc)
        f1s.append(f1)
        print(f"  {cond:<22} AUC={auc:.4f} | F1={f1:.4f}")
 
    print(f"\nMean AUC : {np.mean(aucs):.4f}")
    print(f"Mean F1  : {np.mean(f1s):.4f}")

## Main Run

In [ ]:
if __name__ == "__main__":
    IMAGE_INDEX = build_image_index()
    df = load_multilabel_csv(IMAGE_INDEX)
    _, df_val, df_test = patient_level_split(df)
 
    val_loader = DataLoader(
        ChestXrayDataset(df_val, IMAGE_INDEX, get_eval_transform()),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        ChestXrayDataset(df_test, IMAGE_INDEX, get_eval_transform()),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
    )
 
    model = load_model()
 
    # Optimize thresholds on val set
    val_probs, val_labels = collect_probs(model, val_loader)
    thresholds = find_best_thresholds(val_probs, val_labels)
    threshold_map = dict(zip(ALL_CONDITIONS, thresholds))
 
    with open(THRESHOLD_PATH, "w") as f:
        json.dump(threshold_map, f, indent=2)
 
    print("Per-class thresholds saved:")
    for cond, t in threshold_map.items():
        print(f"  {cond:<22}: {t:.2f}")
 
    # Evaluate on test set
    print("\nTest set evaluation:")
    test_probs, test_labels = collect_probs(model, test_loader)
    evaluate_test(test_probs, test_labels, thresholds)